# 05_01 Borrowed vectors: what did six billion words teach GloVe?

GloVe is a set of word vectors trained at Stanford on about six billion words of Wikipedia and
newswire. Each word is a list of 50 numbers, and nobody chose any of them: they were learned from which
words appear near which. This notebook finds out what that learned, what it can do with arithmetic, and
what it got wrong, including one thing it learned that nobody wanted it to.

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-05-words-as-points-in-space", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'torch': 'torch',
           'pandas': 'pandas',
           'numpy': 'numpy'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json
import os
import numpy as np
from w2vtools import load_glove, nearest
from nlpcheck import ask, guess, reveal, check_05_01

words, index, vectors = load_glove()   # 40,000 words; each row is a unit vector
print(len(words), "words,", vectors.shape[1], "numbers each; the first ten:", words[:10])

## 1. Recall

**r1.** In Lab 02, what cosine similarity did TF-IDF give "a spooky film" and "a scary movie"? (a number)

**r2.** What does skip-gram predict? (a) the centre word from its neighbours, (b) the neighbours from the
centre word, (c) the next sentence

In [ ]:
ask("r1", "")
ask("r2", "")

## 2. A word is a point

Here is the vector for *phone*, and the five words whose vectors point in the most similar direction.
`nearest` computes the cosine between one vector and all 40,000, and sorts.

In [ ]:
print(np.round(vectors[index["phone"]], 2))
for w in ["phone", "refund", "scary"]:
    print(f"{w:8}", nearest(vectors, words, vectors[index[w]], 5, exclude=(w,)))

None of those neighbours was put there by anyone. *Telephone* and *cellphone* sit next to *phone*
because, across six billion words, they are used in the same sentences in the same ways. That is the
whole idea of this chapter, and it has a name, the **distributional hypothesis**: words that occur in
similar contexts have similar meanings.

## 3. Cosine by hand, and a question about opposites

Every vector here has length 1, so the cosine of two words is just their dot product: multiply the 50
pairs of numbers and add them up.

In [ ]:
a, b = vectors[index["scary"]], vectors[index["spooky"]]
print("dot product by hand:", round(float(sum(x * y for x, y in zip(a, b))), 3))
print("the same with @    :", round(float(a @ b), 3))

TF-IDF said *scary* and *spooky* had nothing in common; GloVe says 0.75. Now predict: *hot* and *cold*
mean opposite things. What cosine do you expect between them, somewhere from -1 (opposite directions)
to 1 (the same direction)?

In [ ]:
guess("hot_cold", None)   # a number between -1 and 1

In [ ]:
hot_cold = round(float(vectors[index["hot"]] @ vectors[index["cold"]]), 2)
reveal("hot_cold", hot_cold)
print("good and bad:", round(float(vectors[index["good"]] @ vectors[index["bad"]]), 2))

About 0.8: opposites are **close**. It surprises almost everyone, and it follows directly from how the
vectors were learned. "The water was hot" and "the water was cold" have identical contexts; so do "a
good film" and "a bad film". A model that knows words only by their company cannot tell an antonym from
a synonym, because both keep the same company. This is why a sentiment model built on these vectors
alone struggles with exactly the words sentiment depends on.

## 4. Arithmetic with meaning

If the step from *man* to *king* is the same as the step from *woman* to *queen*, then
`king - man + woman` should land near *queen*. Predict which word comes out nearest.

In [ ]:
guess("king_analogy", None)   # a word, in quotes

In [ ]:
def analogy_worked(a, b, c):
    v = vectors[index[b]] - vectors[index[a]] + vectors[index[c]]
    return nearest(vectors, words, v, 3, exclude=(a, b, c))

answer = analogy_worked("man", "king", "woman")
print(answer)
reveal("king_analogy", answer[0][0])
for a, b, c in [("france", "paris", "italy"), ("walk", "walked", "swim"), ("uk", "london", "canada"), ("slow", "fast", "hot")]:
    print(f"{b} - {a} + {c} ->", analogy_worked(a, b, c))

*Queen*, and *rome*, and *swam* (the past tense, learned without any grammar). Then two failures:
London minus the UK plus Canada gives *sydney*, not Ottawa, and fast minus slow plus hot gives *cool*.
The arithmetic only works for relationships the text states often and consistently; a country's
capital appears far less often than its biggest cities, and "fast is to slow as hot is to cold" is not a
pattern newspapers write. The `exclude` in `analogy_worked` matters too: without it, the nearest word to
`king - man + woman` is usually *king* itself.

## 5. What else the vectors learned

The vectors learned whatever the text contained, not only what anyone would choose to teach. Predict
first: which sense of *bill* will its nearest neighbours show, a charge on an invoice or a law?

In [ ]:
guess("bill_sense", None)   # "invoice" or "law" 

In [ ]:
bill = nearest(vectors, words, vectors[index["bill"]], 6, exclude=("bill",))
print(bill)
reveal("bill_sense", "law")

A law: *legislation*, *senate*, *amendment*. Newswire writes about bills in parliament far more often
than about phone bills, and one vector per word has room for only the average of its senses. For
Kittiwake, whose customers only ever mean the invoice, these borrowed vectors are wrong in the one word
that matters most.

The same mechanism has a harder edge. Run the next cell, which asks the analogy *man is to doctor as
woman is to ...*

In [ ]:
print(analogy_worked("man", "doctor", "woman"))
print(analogy_worked("woman", "nurse", "man"))

The top answer is *nurse*. No one wrote that into GloVe; it absorbed the way occupations and genders
co-occur in the text it was trained on, and it will reproduce that pattern inside any system built on
it: a search engine, a CV screener, a recommendation. This was one of the findings that made the field
take bias in learned representations seriously, and every larger model since has had to be measured for
it. The lesson for your own work is not that vectors are bad; it is that **a model is a portrait of its
training data**, and you have to look at the portrait before you hang it in a product.

Finally, a word GloVe has never seen:

In [ ]:
for w in ["kittiwake", "gullhaven", "esim", "roaming"]:
    print(f"{w:10}", "in the vocabulary" if w in index else "NOT in the vocabulary")

A static vocabulary is fixed when the vectors are trained. New names, products and slang simply do not
exist for it. The subword tokenizers you met in Lab 01 are how modern models get round this: any word
can be spelled from pieces. The next notebook takes the other route and trains vectors on Kittiwake's own
text.

## 6. Your turn

Write the two functions the worked examples used, in your own words, without calling `nearest`:

- `most_similar(word, k)`: the `k` words whose vectors have the highest cosine with `word`'s, **not
  including the word itself**, as a list of strings. Hint: `vectors @ v` gives every cosine at once, and
  `np.argsort(-scores)` sorts the rows from most to least similar.
- `analogy(a, b, c)`: the single word nearest to `b - a + c`, excluding `a`, `b` and `c`.

Then run the save cell and the check.

In [ ]:
def most_similar(word, k=5):
    # YOUR CODE HERE
    return []

def analogy(a, b, c):
    # YOUR CODE HERE
    return None

print(most_similar("phone"), analogy("man", "king", "woman"))

In [ ]:
os.makedirs("out", exist_ok=True)
result = {
    "neighbours": {w: most_similar(w, 5) for w in ["phone", "refund", "spooky"]},
    "analogies": {f"{b} - {a} + {c}": analogy(a, b, c)
                  for a, b, c in [("man", "king", "woman"), ("france", "paris", "italy"), ("uk", "london", "canada")]},
}
json.dump(result, open("out/05_01_glove.json", "w"), indent=1)
check_05_01()

## 7. Exit ticket

Explain it back, in the cell below: why are *hot* and *cold* close in GloVe, and why does that same
reason make the analogy trick work at all? Two sentences.

*Your explanation:* 